<a href="https://colab.research.google.com/github/MightyCrimsonX/Crimson-Notebooks/blob/main/Toriigate_Gradio_Interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Thanks to [Minthy](https://huggingface.co/Minthy/ToriiGate-0.5) for the Gradio interface and [SleepVeryHard](https://huggingface.co/SleepVeryHard/ToriiGate-0.5_GGUF) for the GGUF models.

<img src="https://count.getloli.com/get/@:MghtyToriigate?theme=rule34" height="90px" alt="counted since: Jul 9"/>

In [ ]:
#@title # **Installation and download of models**
#@markdown (Optional) Enter your Huggingface token for faster model downloads.
from IPython.display import clear_output
from IPython.display import display, HTML, Image
hf_token = "" # @param {"type":"string"}
%cd /content
!wget -q https://raw.githubusercontent.com/MightyCrimsonX/Notebook_Scripts/refs/heads/main/scripts/download_magic.py

!pip install aria2

import requests, re

r = requests.get("https://api.github.com/repos/ai-dock/llama.cpp-cuda/releases/latest")
assets = r.json()["assets"]
asset_url = next(a["browser_download_url"] for a in assets if "cuda-12.8-amd64" in a["name"])
print(asset_url)

!wget -q --show-progress -O llamacpp.tar.gz "{asset_url}"
!mkdir -p llama_bin
!tar -xzf llamacpp.tar.gz -C llama_bin
!ls llama_bin
%cd /content/llama_bin
import download_magic
if hf_token:
  !aria2c --console-log-level=warn -c -x 4 -s 4 -k 5M --user-agent="Mozilla/5.0" --header="Authorization: Bearer $hf_token" "https://huggingface.co/SleepVeryHard/ToriiGate-0.5_GGUF/resolve/main/ToriiGate-0.5_Q8_0.gguf" -o ToriiGate-0.5_Q8_0.gguf
  !aria2c --console-log-level=warn -c -x 4 -s 4 -k 5M --user-agent="Mozilla/5.0" --header="Authorization: Bearer $hf_token" "https://huggingface.co/SleepVeryHard/ToriiGate-0.5_GGUF/resolve/main/mmproj_BF16.gguf" -o mmproj_BF16.gguf
else:
  %download https://huggingface.co/SleepVeryHard/ToriiGate-0.5_GGUF/resolve/main/ToriiGate-0.5_Q8_0.gguf
  %download https://huggingface.co/SleepVeryHard/ToriiGate-0.5_GGUF/resolve/main/mmproj_BF16.gguf

clear_output()
display(Image(url='https://media1.giphy.com/media/v1.Y2lkPTc5MGI3NjExd3hzeHZ6bGE2YjF5YzN3MXp1NmFmdGo2ODlpN3Y2NGZhOWlxcXdlZSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/fybt3q7jokUbS/giphy.gif'))
display(HTML(f"<h1 style='color: yellow;'>Installation complete</h1>"))

In [ ]:
#@markdown # **Start Gradio Toriigate Interface**
#@markdown Model loading time: 1-2 minutes
%cd /content/llama_bin/cuda-12.8/

!wget -q -nc https://huggingface.co/Minthy/ToriiGate-0.5/resolve/main/scripts/gradio_interface.py
!wget -q -nc https://huggingface.co/Minthy/ToriiGate-0.5/resolve/main/scripts/caption_distributed.py
!wget -q -nc https://huggingface.co/Minthy/ToriiGate-0.5/resolve/main/scripts/prompts.py
!pip install -q gradio

import subprocess, time, requests


llama_proc = subprocess.Popen([
    "./llama-server",
    "--model", "/content/llama_bin/ToriiGate-0.5_Q8_0.gguf",
    "--mmproj", "/content/llama_bin/mmproj_BF16.gguf",
    "--alias", "ToriiGate-0.5",
    "-ngl", "99",
    "--port", "7801"
])


print("Waiting for llama-server to load the model...")
for _ in range(60):
    try:
        r = requests.get("http://127.0.0.1:7801/health", timeout=2)
        if r.status_code == 200:
            print("✅ llama-server ready")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2)
else:
    print("⚠️ It took too long, check if the process died")


with open("gradio_interface.py", "r") as f:
    content = f.read()

content = content.replace(
    'API_URL = "http://127.0.0.1:8000/v1/chat/completions"',
    'API_URL = "http://127.0.0.1:7801/v1/chat/completions"'
)
content = content.replace(
    'app.launch(server_name="127.0.0.1", server_port=7860)',
    'app.launch(server_name="0.0.0.0", server_port=7860, share=True)'
)

with open("gradio_interface.py", "w") as f:
    f.write(content)

!python gradio_interface.py